# Modelo pedidos digitales

In [0]:
!pip install xgboost hyperopt category_encoders

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import col, lit, udf
from pyspark.sql.window import Window

In [0]:
spark = SparkSession.builder.appName("Modelo Pedidos Digitales").getOrCreate()

In [0]:
df = spark.read.table("workspace.default.transaccional_clientes")
display(df.limit(10))

In [0]:
id_cols = ['cliente_id', 'agencia_id', 'ruta_id']

cat_cols = ['pais', 'region_comercial', 'tipo_cliente', 'madurez_digital', 'frecuencia_visitas', 'canal_pedido']

disc_cols = ['estrellas', 'materiales_distintos']

num_cols = ['facturacion_usd', 'cajas_fisicas']

date_cols = ['fecha_pedido_dt']

## Distribuciones

In [0]:
for col_name in num_cols:
    data = df.select(col_name).dropna().toPandas()
    plt.figure(figsize=(8, 4))
    sns.histplot(data[col_name], kde=True, bins=30)
    plt.title(f'Distribución de {col_name}')
    plt.xlabel(col_name)
    plt.ylabel('Frecuencia')
    plt.show()

In [0]:
for col_name in disc_cols:
    total_count = df.filter(col(col_name).isNotNull()).count()
    count_df = (
        df.groupBy(col_name)
        .agg(f.count("*").alias("count"))
        .withColumn("percentage", col("count") / total_count)
        .orderBy(col(col_name).asc())
    )
    display(count_df)

In [0]:
for col_name in cat_cols:
    total_count = df.filter(col(col_name).isNotNull()).count()
    count_df = (
        df.groupBy(col_name)
        .agg(f.count("*").alias("count"))
        .withColumn("percentage", col("count") / total_count)
        .orderBy(col("count").desc())
    )
    display(count_df)

## Análisis de Datos

In [0]:
cliente_canal = (
    df
    .groupBy("cliente_id", "canal_pedido")
    .agg(f.count("*").alias("count"))
)

total_por_cliente = df.groupBy("cliente_id").agg(f.count("*").alias("total"))

cliente_canal = (
    cliente_canal
    .join(total_por_cliente, on="cliente_id", how="left")
    .withColumn("percentage", col("count") / col("total"))
    .select("cliente_id", "canal_pedido", "count", "percentage")
)

# Convert to pandas for plotting
cliente_canal_pd = cliente_canal.toPandas()

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=cliente_canal_pd,
    x="percentage",
    hue="canal_pedido"
)
plt.title("Histograma del porcentaje por canal_pedido")
plt.xlabel("Porcentaje")
plt.ylabel("Frecuencia")
plt.legend(title="Canal Pedido")
plt.show()

In [0]:
df_ym = df.withColumn(
    "year_month",
    f.date_format(col("fecha_pedido"), "yyyyMM").cast("int")
)

cliente_min_max = (
    df_ym.groupBy("cliente_id")
    .agg(
        f.min("year_month").alias("min_year_month"),
        f.max("year_month").alias("max_year_month"),
        f.countDistinct("year_month").alias("meses_activo")
    )
)

cliente_min_max = (
    cliente_min_max
    .withColumn("min_date", f.to_date(f.concat_ws("-", col("min_year_month").cast("string").substr(1,4), col("min_year_month").cast("string").substr(5,2), f.lit("01"))))
    .withColumn("max_date", f.to_date(f.concat_ws("-", col("max_year_month").cast("string").substr(1,4), col("max_year_month").cast("string").substr(5,2), f.lit("01"))))
    .withColumn("meses_total", f.months_between(col("max_date"), col("min_date")) + 1)
    .withColumn("meses_total", col("meses_total").cast("int"))
    .select("cliente_id", "min_year_month", "max_year_month", "meses_total", "meses_activo")
)

display(cliente_min_max.limit(20))

In [0]:
import random
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col

# ----------------------------
# Sample 10 clientes aleatorios
# ----------------------------
clientes_all = [row['cliente_id'] for row in df.select("cliente_id").distinct().collect()]
n_clients = min(10, len(clientes_all))
clientes_sample = random.sample(clientes_all, n_clients)

# ----------------------------
# Ventana y columnas calculadas
# ----------------------------
wind_cliente_seq = Window.partitionBy("cliente_id").orderBy("fecha_pedido")

df_seq = (
    df.filter(col("cliente_id").isin(clientes_sample))
    .withColumn("nro_pedido", F.row_number().over(wind_cliente_seq))
    .withColumn("es_digital", (col("canal_pedido") == "DIGITAL").cast("int"))
    .select("cliente_id", "nro_pedido", "es_digital")
    .orderBy("cliente_id", "nro_pedido")
)

# ----------------------------
# Convertir a pandas
# ----------------------------
df_seq_pd = df_seq.toPandas()

# ----------------------------
# Crear subplots por cliente
# ----------------------------
num_clientes = len(clientes_sample)
fig, axes = plt.subplots(num_clientes, 1, figsize=(12, 3*num_clientes), sharex=True)

if num_clientes == 1:
    axes = [axes]  # asegurar que axes sea iterable si solo hay un cliente

for ax, cliente in zip(axes, clientes_sample):
    data = df_seq_pd[df_seq_pd['cliente_id'] == cliente]
    ax.plot(
        data['nro_pedido'],
        data['es_digital'],
        marker='o',
        drawstyle='steps-mid',
        color='tab:blue'
    )
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['NO DIGITAL', 'DIGITAL'])
    ax.set_title(f'Cliente {cliente}')
    ax.set_ylabel('Canal Digital')

axes[-1].set_xlabel('Número de Pedido')
plt.tight_layout()
plt.show()



## Creación del dataset

### Target

Creamos nuestro target como el canal del siguiente pedido. Eliminamos los ultimos pedidos de cada cliente, al no saber cual será su siguiente canal.

In [0]:
wind_cliente_p = Window.partitionBy("cliente_id").orderBy("fecha_pedido")

df_target = df.withColumn(
    "digital_target",
    (f.lead("canal_pedido").over(wind_cliente_p) == "DIGITAL").cast("int")
).filter(col("digital_target").isNotNull())

print('Dataset original:', df.count())
print('Dataset con target:', df_target.count())

display(df_target.limit(10))

### Variables de ultimos pedidos

In [0]:
wind_cliente_p_3 = wind_cliente_p.rowsBetween(-2, 0)
wind_cliente_p_6 = wind_cliente_p.rowsBetween(-5, 0)
wind_cliente_p_hist = wind_cliente_p.rowsBetween(Window.unboundedPreceding, 0)

df_pedidos_vars = (
    df_target
    .withColumn(
        "pct_digital_ult_3",
        (f.sum((col("canal_pedido") == "DIGITAL").cast("int")).over(wind_cliente_p_3)) / (f.count("*").over(wind_cliente_p_3))
    )
    .withColumn(
        "pct_telefono_ult_3",
        (f.sum((col("canal_pedido") == "TELEFONO").cast("int")).over(wind_cliente_p_3)) / (f.count("*").over(wind_cliente_p_3))
    )
    .withColumn(
        "pct_vendedor_ult_3",
        (f.sum((col("canal_pedido") == "VENDEDOR").cast("int")).over(wind_cliente_p_3)) / (f.count("*").over(wind_cliente_p_3))
    )
    .withColumn(
        "pct_digital_ult_6",
        (f.sum((col("canal_pedido") == "DIGITAL").cast("int")).over(wind_cliente_p_6)) / (f.count("*").over(wind_cliente_p_6))
    )
    .withColumn(
        "pct_telefono_ult_6",
        (f.sum((col("canal_pedido") == "TELEFONO").cast("int")).over(wind_cliente_p_6)) / (f.count("*").over(wind_cliente_p_6))
    )
    .withColumn(
        "pct_vendedor_ult_6",
        (f.sum((col("canal_pedido") == "VENDEDOR").cast("int")).over(wind_cliente_p_6)) / (f.count("*").over(wind_cliente_p_6))
    )
    .withColumn(
        "dias_desde_ultimo_pedido",
        (f.unix_timestamp("fecha_pedido") - f.unix_timestamp(f.lag("fecha_pedido").over(wind_cliente_p))) / 86400
    )
    .withColumn(
        "avg_dias_entre_pedidos_ult_6",
        f.when(
            f.count("fecha_pedido").over(wind_cliente_p_6) >= 2,  # mínimo 2 pedidos
            f.avg("dias_desde_ultimo_pedido").over(wind_cliente_p_6)
        )
    )
    .withColumn(
        "monto_total_ult_6",
        f.sum("facturacion_usd").over(wind_cliente_p_6)
    )
    .withColumn(
        "avg_materiales_distintos_ult_6",
        f.avg("materiales_distintos").over(wind_cliente_p_6)
    )
    .withColumn(
        "suma_cajas_fisicas_ult_6",
        f.sum("cajas_fisicas").over(wind_cliente_p_6)
    )
    .withColumn(
        "cantidad_pedidos_historicos",
        f.count("*").over(wind_cliente_p_hist)
    )
)

display(df_pedidos_vars.limit(50))

In [0]:
df_final = df_pedidos_vars.dropna()

print('Dataset original:', df.count())
print('Dataset con target:', df_target.count())
print('Dataset con variables:', df_pedidos_vars.count())
print('Dataset final:', df_final.count())

## Desarrollo del Modelo

In [0]:
import mlflow
import mlflow.xgboost
from sklearn.model_selection import train_test_split, cross_val_score
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score, roc_auc_score,
    log_loss, roc_curve
)
from scipy.stats import ks_2samp
from xgboost import XGBClassifier
import numpy as np

In [0]:
total_count = df_final.count()
digital_count = df_final.filter(col("digital_target") == 1).count()
no_digital_count = total_count - digital_count

percentage_df = spark.createDataFrame([
    ("DIGITAL", digital_count, digital_count / total_count),
    ("NO_DIGITAL", no_digital_count, no_digital_count / total_count)
], ["tipo", "count", "percentage"])

display(percentage_df)

In [0]:
df_pd = df_final.toPandas()

cat_cols = [
    'pais', 'region_comercial', 'tipo_cliente', 'madurez_digital',
    'frecuencia_visitas', 'canal_pedido'
]

for col in cat_cols:
    df_pd[col] = df_pd[col].astype('category')

features = [col for col in df_pd.columns if col not in ['digital_target', 'cliente_id', 'fecha_pedido']]
X = df_pd[features]
y = df_pd['digital_target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=24, stratify=y)


In [0]:
from category_encoders import TargetEncoder

encoder = TargetEncoder(cols=['agencia_id', 'ruta_id'])
X_train[['agencia_id', 'ruta_id']] = encoder.fit_transform(X_train[['agencia_id', 'ruta_id']], y_train)
X_test[['agencia_id', 'ruta_id']] = encoder.transform(X_test[['agencia_id', 'ruta_id']])

model = XGBClassifier(
    random_state=24,
    enable_categorical=True,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

# Predicciones
y_train_pred = model.predict(X_train)
y_train_pred_proba = model.predict_proba(X_train)[:, 1]
y_test_pred = model.predict(X_test)
y_test_pred_proba = model.predict_proba(X_test)[:, 1]

metrics_df = pd.DataFrame([
    {
        "set": "train",
        "accuracy": accuracy_score(y_train, y_train_pred),
        "f1_score": f1_score(y_train, y_train_pred),
        "recall": recall_score(y_train, y_train_pred),
        "precision": precision_score(y_train, y_train_pred),
        "aucroc": roc_auc_score(y_train, y_train_pred_proba),
        "ks": ks_2samp(y_train, y_train_pred_proba)[0]
    },
    {
        "set": "test",
        "accuracy": accuracy_score(y_test, y_test_pred),
        "f1_score": f1_score(y_test, y_test_pred),
        "recall": recall_score(y_test, y_test_pred),
        "precision": precision_score(y_test, y_test_pred),
        "aucroc": roc_auc_score(y_test, y_test_pred_proba),
        "ks": ks_2samp(y_test, y_test_pred_proba)[0]
    }
])

display(metrics_df)

In [0]:
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
feature_names = X_train.columns

plt.figure(figsize=(10, 6))
plt.title("Feature Importances")
plt.bar(range(len(importances)), importances[indices], align="center")
plt.xticks(range(len(importances)), feature_names[indices], rotation=90)
plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(10, 6))

sns.histplot(
    y_test_pred_proba[y_test == 0],
    color="red",
    label="No Digitales",
    stat="probability",
    bins=30,
    alpha=0.5
)

sns.histplot(
    y_test_pred_proba[y_test == 1],
    color="blue",
    label="Digitales",
    stat="probability",
    bins=30,
    alpha=0.3
)

plt.title("Distribución porcentual de probabilidades: Digitales vs No Digitales")
plt.xlabel("Probabilidad predicha de ser Digital")
plt.ylabel("Porcentaje de la población")
plt.legend()
plt.tight_layout()
plt.show()